# Phase 7 & 8: Model Evaluation and Generalization

This notebook evaluates the trained deepfake audio detection classifiers on the testing set. It computes standard classification metrics, details the Equal Error Rate (EER), and measures model robustness against simulated environmental and channel distortions.

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score
from src.train import EmbeddingDataset
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_roc_curve, calculate_eer
from src.evaluate import apply_reverberation, apply_bandpass_filter, apply_quantization
from src.preprocess import preprocess_audio_tensor, preprocess_audio
from src.embeddings import HubertExtractor
from src.model import BaselineMLP, AttentionMLP, BiLSTMClassifier

# Aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Test Dataset and Models

We instantiate the test loader and load the checkpoints for our trained classifiers.

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
proj_dir = 'C:/Users/Shreshtha Shrinivas/.gemini/antigravity/scratch/deepfake-audio-detection'
metadata_path = os.path.join(proj_dir, 'data', 'metadata_subset.csv')

test_dataset = EmbeddingDataset(metadata_path, 'testing', proj_dir)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f"Loaded test set with {len(test_dataset)} samples.")

Loaded test set with 200 samples.


## 2. Evaluate Classifiers

In [3]:
def load_checkpoint(model_type):
    checkpoint_path = os.path.join(proj_dir, 'models', f'best_{model_type}.pt')
    if not os.path.exists(checkpoint_path):
        print(f"Warning: Checkpoint not found for {model_type}")
        return None
    checkpoint = torch.load(checkpoint_path, map_location=device)
    input_dim = checkpoint.get('input_dim', 2304)
    
    if model_type == 'baseline':
        model = BaselineMLP(input_dim=input_dim)
    elif model_type == 'attention':
        model = AttentionMLP(input_dim=input_dim)
    elif model_type == 'lstm':
        model = BiLSTMClassifier(input_dim=input_dim, lstm_hidden=128)
        
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    return model

models = {}
for m_type in ['baseline', 'attention', 'lstm']:
    loaded = load_checkpoint(m_type)
    if loaded:
        models[m_type] = loaded

In [4]:
results = []
eval_data = {}

for name, model in models.items():
    print(f"Evaluating {name}...")
    metrics, y_true, y_pred, y_scores = evaluate_model(model, test_loader, device)
    
    # Save predictions for plots
    eval_data[name] = {
        'metrics': metrics,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_scores': y_scores
    }
    
    results.append({
        'Model': name.upper(),
        'Accuracy': f"{metrics['accuracy']*100:.2f}%",
        'F1 Score': f"{metrics['f1']*100:.2f}%",
        'Precision': f"{metrics['precision']*100:.2f}%",
        'Recall': f"{metrics['recall']*100:.2f}%",
        'EER': f"{metrics['eer']*100:.2f}%",
        'Per-Class Real Acc': f"{metrics['real_accuracy']*100:.2f}%",
        'Per-Class Fake Acc': f"{metrics['fake_accuracy']*100:.2f}%"
    })

df_results = pd.DataFrame(results)
df_results

Evaluating baseline...


Evaluating attention...


Evaluating lstm...


,Model,Accuracy,F1 Score,Precision,Recall,EER,Per-Class Real Acc,Per-Class Fake Acc
0,BASELINE,80.50%,78.21%,88.61%,70.00%,19.50%,91.00%,70.00%
1,ATTENTION,80.00%,75.31%,98.39%,61.00%,12.00%,99.00%,61.00%
2,LSTM,77.00%,76.77%,77.55%,76.00%,22.00%,78.00%,76.00%


## 3. Generate Evaluation Figures for Best Model

We pick the model with the lowest Equal Error Rate (EER) and plot its ROC Curve and Confusion Matrix.

In [5]:
# Select best model type
if eval_data:
    best_model_name = min(eval_data.keys(), key=lambda k: eval_data[k]['metrics']['eer'])
    print(f"Best model by EER: {best_model_name.upper()}")
    
    best_metrics = eval_data[best_model_name]['metrics']
    best_y_true = eval_data[best_model_name]['y_true']
    best_y_scores = eval_data[best_model_name]['y_scores']
    best_y_pred = eval_data[best_model_name]['y_pred']
    
    # Generate and save Confusion Matrix
    cm_path = os.path.join(proj_dir, 'reports', 'confusion_matrix.png')
    plot_confusion_matrix(best_metrics['confusion_matrix'], cm_path)
    print(f"Confusion matrix saved to {cm_path}")
    
    # Generate and save ROC Curve
    roc_path = os.path.join(proj_dir, 'reports', 'roc_curve.png')
    plot_roc_curve(best_y_true, best_y_scores, best_metrics['eer'], roc_path)
    print(f"ROC curve saved to {roc_path}")
else:
    print("No models evaluated.")

Best model by EER: ATTENTION


Confusion matrix saved to C:/Users/Shreshtha Shrinivas/.gemini/antigravity/scratch/deepfake-audio-detection\reports\confusion_matrix.png


ROC curve saved to C:/Users/Shreshtha Shrinivas/.gemini/antigravity/scratch/deepfake-audio-detection\reports\roc_curve.png


## 4. Phase 8: Robustness and Channel Distortion Simulation

To measure how our model generalizes, we simulate physical rerecording, channel bandpass filtering, and PCM digital compression.

In [6]:
# We perform evaluation by processing audio waveforms directly
# We load the test subset metadata, load and transform each audio, run it through HuBERT on the fly, and then feed it into the best classifier model
df_test = pd.read_csv(metadata_path)
df_test = df_test[df_test['split'] == 'testing'].reset_index(drop=True)

# Limit to 100 samples for fast evaluation on CPU
df_test_sample = df_test.sample(n=min(100, len(df_test)), random_state=42).reset_index(drop=True)
print(f"Evaluating channel distortion on balanced sample size: {len(df_test_sample)}")

Evaluating channel distortion on balanced sample size: 100


In [7]:
# Load HuBERT extractor
extractor = HubertExtractor(device=device)
best_classifier = models[best_model_name]
best_classifier.eval()

def run_robustness_eval(transform_fn=None, name="Clean"):
    all_labels = []
    all_scores = []
    all_preds = []
    
    label_map = {'real': 0, 'fake': 1}
    
    for _, row in df_test_sample.iterrows():
        file_path = os.path.join(proj_dir, row['filepath'])
        label = label_map[row['label']]
        
        # 1. Preprocess
        waveform = preprocess_audio(file_path)
        
        # 2. Apply distortion
        if transform_fn is not None:
            waveform = transform_fn(waveform)
            
        # 3. Extract embeddings
        waveform_tensor = torch.tensor(waveform, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            feats = extractor.extract_features(waveform_tensor)
            embedding = feats['combined'].unsqueeze(0).to(device)
            logits = best_classifier(embedding)
            probs = torch.softmax(logits, dim=1)
            
        all_labels.append(label)
        all_scores.append(probs[0, 1].item())
        all_preds.append(torch.argmax(logits, dim=1).item())
        
    y_true = np.array(all_labels)
    y_scores = np.array(all_scores)
    y_pred = np.array(all_preds)
    
    acc = accuracy_score(y_true, y_pred)
    eer, _ = calculate_eer(y_true, y_scores)
    
    return acc, eer

print("Running robustness simulations...")
acc_clean, eer_clean = run_robustness_eval(None, "Clean")
print(f"Clean: Acc = {acc_clean*100:.2f}%, EER = {eer_clean*100:.2f}%")

acc_reverb, eer_reverb = run_robustness_eval(apply_reverberation, "Reverb")
print(f"Reverb: Acc = {acc_reverb*100:.2f}%, EER = {eer_reverb*100:.2f}%")

acc_tele, eer_tele = run_robustness_eval(lambda w: apply_bandpass_filter(w, lowcut=300, highcut=3400), "Telephone")
print(f"Telephone Filter: Acc = {acc_tele*100:.2f}%, EER = {eer_tele*100:.2f}%")

acc_quant, eer_quant = run_robustness_eval(lambda w: apply_quantization(w, bits=8), "8-bit PCM")
print(f"8-bit PCM Compression: Acc = {acc_quant*100:.2f}%, EER = {eer_quant*100:.2f}%")

Loading HuBERT model 'facebook/hubert-base-ls960' on cpu...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Running robustness simulations...


Clean: Acc = 77.00%, EER = 10.06%


Reverb: Acc = 54.00%, EER = 33.90%


Telephone Filter: Acc = 87.00%, EER = 15.02%


8-bit PCM Compression: Acc = 56.00%, EER = 40.10%


### Summary of Distortion Robustness

In [8]:
robust_results = [
    {'Channel Condition': 'Clean (Unmodified)', 'Accuracy': f'{acc_clean*100:.2f}%', 'EER': f'{eer_clean*100:.2f}%'},
    {'Channel Condition': 'Simulated Reverberation', 'Accuracy': f'{acc_reverb*100:.2f}%', 'EER': f'{eer_reverb*100:.2f}%'},
    {'Channel Condition': 'Telephone Bandpass Filter (300-3400Hz)', 'Accuracy': f'{acc_tele*100:.2f}%', 'EER': f'{eer_tele*100:.2f}%'},
    {'Channel Condition': '8-bit PCM Compression', 'Accuracy': f'{acc_quant*100:.2f}%', 'EER': f'{eer_quant*100:.2f}%'}
]
df_robust = pd.DataFrame(robust_results)
df_robust.to_csv(os.path.join(proj_dir, 'reports', 'robustness_results.csv'), index=False)
df_robust

,Channel Condition,Accuracy,EER
0,Clean (Unmodified),77.00%,10.06%
1,Simulated Reverberation,54.00%,33.90%
2,Telephone Bandpass Filter (300-3400Hz),87.00%,15.02%
3,8-bit PCM Compression,56.00%,40.10%
